# Environment Setup
Uncomment lines below and run them once to prepare environment.

In [130]:
# python -m venv .venv
# # source .venv/bin/activate
# sourec .venv\Scripts\activate # Windows
# pip install jupyterlab pandas scikit-learn pyarrow matplotlib
# # jupyter lab

# Data preparation
## Reading files

In [131]:
import pandas as pd

files_list = [
    "data/items.sell.csv",
    "data/items.change.csv",
    "data/items.free.csv"
]

files_key_columns = ['name', 'tag']

dfs = [pd.read_csv(file, usecols=files_key_columns) for file in files_list]

df = pd.concat(dfs, ignore_index=True)
df.tail(10)

,name,tag
290047,Shimano Saint Piasta Tył 8/9rz 36H FH-M810 Ham...,Sport i Hobby » Rowery
290048,Części do Galaxy S4 I9505,Telefony komórkowe » Samsung
290049,Cube Reaction carbon gwarancja,Sport i Hobby » Rowery
290050,miś maskotka,Dla Dzieci » Zabawki
290051,Mouille nowy strój kąpielowy jednoczęsciowy gr...,Moda » Bielizna
290052,miętowa torba na ramię,Moda » Dodatki
290053,Bardzo dobry stan kia k2700,Dostawcze i Ciężarowe » Dostawcze
290054,Ławka Treningowa + gryf,Sport i Hobby » Fitness
290055,Kurtka Wiosenna r.134/140 jak Zara H&amp;M,Dla Dzieci » Ubranka dla dzieci
290056,Zderzak BMW E38 oryginalny lewa połowa,Części samochodowe » Osobowe


## Cleaning and normalization
### Missing row values
Removing rows with missing values.
The value `axis=0` applies to table rows.

In [132]:
print(df.isna().sum())
df = df.dropna(axis=0)

name    0
tag     0
dtype: int64


### Whitespaces
Replacing all whitespace characters with a single space and droping those rows where empty string left.

In [133]:
df["name"] = df["name"].str.replace(r"\s+", " ", regex=True).str.strip()
df["tag"] = df["tag"].str.replace(r"\s+", " ", regex=True).str.strip()

df = df[(df["name"] != "") & (df["tag"] != "")]

### Duplicates
Droping duplicated rows.

In [134]:
print("duplicates ('name'+'tag'):", df.duplicated().sum())
df = df.drop_duplicates()

print(df.shape)

duplicates ('name'+'tag'): 63564
(226493, 2)


## Categories and subcategories
Spliting `tag` column to `category` and `subcategory`.
They are separated by ` » `.

In [135]:
parts = df["tag"].str.split(" » ")
print(parts.str.len().value_counts()) # type: ignore

df["category"] = parts.str[0]
df["subcategory"] = parts.str[1]

df.tail(10)

tag
2    226493
Name: count, dtype: int64


,name,tag,category,subcategory
289586,Oddam obornik kurzy,Rolnictwo » Ryneczek,Rolnictwo,Ryneczek
289587,Tunika z dzianiny rozm.74,Dla Dzieci » Ubranka dla dzieci,Dla Dzieci,Ubranka dla dzieci
289588,Bluzy za darmo,Dla Dzieci » Ubranka dla dzieci,Dla Dzieci,Ubranka dla dzieci
289589,Obrazki Matki Bożej Pocieszenia z Przeworska,Sport i Hobby » Kolekcje,Sport i Hobby,Kolekcje
289590,ZAPLACE za wywóz. Ziemia z budowy piach kamien...,Dom i Ogród » Materiały budowlane,Dom i Ogród,Materiały budowlane
289592,Zestaw książek dla dzieci,Książki » Poradniki i albumy,Książki,Poradniki i albumy
289593,biała bluzka wizytowa rozm 36 ZA DARMO,Moda » Ubrania,Moda,Ubrania
289594,Kołderka z poszewka,Dla Dzieci » Pozostałe dla dzieci,Dla Dzieci,Pozostałe dla dzieci
289596,Akumulator 61Ah Varta Silver Promocja,Części samochodowe » Osobowe,Części samochodowe,Osobowe
289597,Doradca Klienta Biznesowego / Detalicznego 300...,Praca » Obsługa klienta / call center,Praca,Obsługa klienta / call center


# Conflicts

## Analysys

In [136]:
conflicts = df.groupby("name")["tag"].nunique()
print("names with more than 1 diferent tags:", (conflicts > 1).sum())
df[df["name"].isin(conflicts[conflicts > 1].index[:5])].sort_values("name")

names with more than 1 diferent tags: 1305


,name,tag,category,subcategory
251354,&quot;Zamienie&quot; OKAZJA!!!,Elektronika » Pozostała elektronika,Elektronika,Pozostała elektronika
251358,&quot;Zamienie&quot; OKAZJA!!!,Sport i Hobby » Pozostały sport i hobby,Sport i Hobby,Pozostały sport i hobby
195360,Adidasy,Dla Dzieci » Buciki,Dla Dzieci,Buciki
260103,Adidasy,Moda » Buty,Moda,Buty
60155,Agregat prądotwórczy,Usługi i Firmy » Wyposażenie firm,Usługi i Firmy,Wyposażenie firm
152002,Agregat prądotwórczy,Dom i Ogród » Narzędzia,Dom i Ogród,Narzędzia
71325,Akademia Pana Kleksa,Książki » Dla dzieci,Książki,Dla dzieci
82073,Akademia Pana Kleksa,Książki » Literatura,Książki,Literatura
104030,Akcesoria ślubne,Moda ślubna » Suknie ślubne,Moda ślubna,Suknie ślubne
107568,Akcesoria ślubne,Akcesoria ślubne » Dekoracje ślubne,Akcesoria ślubne,Dekoracje ślubne


In [137]:
conf_names = conflicts[conflicts > 1].index
conf_df = df[df["name"].isin(conf_names)]

(conf_df.groupby("name")["tag"]
    .apply(lambda s: " | ".join(sorted(s.unique())))
    .value_counts().head(15))

tag
Dla Dzieci » Ubranka dla dzieci | Moda » Ubrania                        76
Usługi i Firmy » Budowa i Remont | Usługi i Firmy » Pozostałe usługi    24
Sport i Hobby » Kolekcje | Wyposażenie wnętrz » Dekoracje               23
Moda ślubna » Pozostałe | Moda ślubna » Suknie ślubne                   21
Rolnictwo » Produkty rolne | Rolnictwo » Ryneczek                       21
Dla Dzieci » Buciki | Moda » Buty                                       20
Książki » Książki naukowe | Książki » Podręczniki szkolne               17
Moda ślubna » Buty ślubne | Moda ślubna » Suknie ślubne                 13
Dla Dzieci » Buciki | Sport i Hobby » Obuwie sportowe                   12
Nieruchomości » Noclegi | Nieruchomości » Stancje i Pokoje              11
Mieszkania » Wynajem | Nieruchomości » Noclegi                          10
Giełda zwierząt » Drób | Zwierzęta » Ptaki                              10
Meble » Pozostałe meble | Meble » Szafy i komody                        10
Rolnictwo » Pozostałe

## Cleaning
Most conflicts are caused by the same item being added to multiple categories. Removing them.

In [138]:
df = df[~df["name"].isin(conf_names)]
print(df.shape)

(223239, 4)


# Class distribution

In [139]:
counts = df["tag"].value_counts()
for t in (20, 50, 100, 200):
    kept = counts[counts >= t]
    print(f"min {t:>3}: classes {len(kept):>3}, preserved samples {kept.sum() / counts.sum():.1%}")

min  20: classes 259, preserved samples 99.7%
min  50: classes 227, preserved samples 99.3%
min 100: classes 191, preserved samples 98.2%
min 200: classes 160, preserved samples 96.1%


In [140]:
MIN_SAMPLES = 50

## Tags
Full - more useful, but with a long tail of rare classes.

In [141]:
tag_col_name = "tag"
df_tag_col = df["tag"]
tag_col_value_counts = df["tag"].value_counts()
tag_col_value_counts

tag
Elektronika » Gry i Konsole                18112
Dla Dzieci » Ubranka dla dzieci            12059
Rolnictwo » Części do maszyn rolniczych     7737
Moda ślubna » Suknie ślubne                 7097
Części samochodowe » Osobowe                6682
                                           ...  
Praca » HR                                     1
Praca » Ochrona                                1
Gastronomia » Pomoc kuchenna / zmywak          1
Praca » Marketing / PR / media                 1
Praca » Finanse / księgowość                   1
Name: count, Length: 342, dtype: int64

In [142]:
tag_col_value_counts.describe()

count      342.000000
mean       652.745614
std       1560.090108
min          1.000000
25%         23.000000
50%        162.000000
75%        600.500000
max      18112.000000
Name: count, dtype: float64

In [143]:
(tag_col_value_counts < MIN_SAMPLES).sum()

np.int64(115)

## Categories
Main level - few classes, high effectiveness.

In [144]:
category_col_name = "category"
df_category_col = df["category"]
category_col_value_counts = df["category"].value_counts()
category_col_value_counts

category
Elektronika                     26085
Dla Dzieci                      20804
Sport i Hobby                   19161
Usługi i Firmy                  18741
Rolnictwo                       14980
Książki                         14791
Moda ślubna                      9466
Dom i Ogród                      8718
Zwierzęta                        8415
Części samochodowe               7833
Wyposażenie wnętrz               6596
Mieszkania                       6149
Komputery                        5192
Moda                             5161
Samochody osobowe                4425
Meble                            3983
Psy                              3931
Akwarystyka                      3927
Muzyka i Edukacja                3778
Opony i Felgi                    3487
Telefony komórkowe               3447
Biura i Lokale                   3305
Nieruchomości                    3057
Usługi ślubne                    2704
Maszyny rolnicze                 2433
Motoryzacja                      1878
Dom

In [145]:
category_col_value_counts.describe()

count       41.000000
mean      5444.853659
std       6413.776487
min          8.000000
25%        875.000000
50%       3447.000000
75%       6596.000000
max      26085.000000
Name: count, dtype: float64

In [146]:
(category_col_value_counts < MIN_SAMPLES).sum()

np.int64(1)

# Data selection

> ✋ Choose selected data by commenting right line

In [147]:
(col_name, df_col, col_value_counts) = (tag_col_name, df_tag_col, tag_col_value_counts)
# (col_name, df_col, col_value_counts) = (category_col_name, df_category_col, category_col_value_counts)


selected_col_values = col_value_counts[col_value_counts >= MIN_SAMPLES].index

df_final = df[df_col.isin(selected_col_values)]
df_final.tail(10)


,name,tag,category,subcategory
289584,"Zasilacz 7,5 v 1,5 A",Elektronika » Akcesoria telefoniczne,Elektronika,Akcesoria telefoniczne
289585,kamizelka M,Moda » Ubrania,Moda,Ubrania
289586,Oddam obornik kurzy,Rolnictwo » Ryneczek,Rolnictwo,Ryneczek
289587,Tunika z dzianiny rozm.74,Dla Dzieci » Ubranka dla dzieci,Dla Dzieci,Ubranka dla dzieci
289589,Obrazki Matki Bożej Pocieszenia z Przeworska,Sport i Hobby » Kolekcje,Sport i Hobby,Kolekcje
289590,ZAPLACE za wywóz. Ziemia z budowy piach kamien...,Dom i Ogród » Materiały budowlane,Dom i Ogród,Materiały budowlane
289592,Zestaw książek dla dzieci,Książki » Poradniki i albumy,Książki,Poradniki i albumy
289593,biała bluzka wizytowa rozm 36 ZA DARMO,Moda » Ubrania,Moda,Ubrania
289594,Kołderka z poszewka,Dla Dzieci » Pozostałe dla dzieci,Dla Dzieci,Pozostałe dla dzieci
289596,Akumulator 61Ah Varta Silver Promocja,Części samochodowe » Osobowe,Części samochodowe,Osobowe


# Sets preparation for training

In [148]:
from sklearn.model_selection import train_test_split

# train - training set: data used to train the model (learn)
# val - validation set: data used to evaluate and tune the model during development (tune)
# test – test set: data used for the final evaluation of the trained model (final evaluation)

# 80/20
train, temp = train_test_split(
    df_final,
    test_size=0.2,
    stratify=df_final[col_name], #df_col
    random_state=42)

# 50/50
val, test = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp[col_name],
    random_state=42)

train.to_parquet("data/train.parquet", index=False)
val.to_parquet("data/val.parquet", index=False)
test.to_parquet("data/test.parquet", index=False)

print(len(train), len(val), len(test))

177294 22162 22162
